# Evaluation Dataset Analysis

This notebook explores and processes the evaluation dataset stored in `goi_search_results.json`.

The goal is to:
- Understand the structure of the dataset
- Extract and organize query and result-level data
- Transform nested JSON into tabular format
- Export clean datasets for further analysis

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")

In [2]:
import requests
import json
import pandas as pd
import re

## Understanding the Dataset Structure

Loading the JSON file and inspecting its structure.

This step helps identify:
- The overall format of the data (list or dictionary)
- Number of entries (queries)
- Structure of individual records

Also examining the first entry to understand the schema of the dataset.

In [3]:
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)

# Understand the structure
print(type(data))
print(f"Number of entries: {len(data)}")

# Look at the first entry
print("\nFirst entry:")
print(json.dumps(data[0], indent=2))

<class 'list'>
Number of entries: 101

First entry:
{
  "query": "software companies",
  "query_id": 1,
  "total_results": 1000,
  "results": [
    {
      "rank": 1,
      "similarity_score": 0.7662015,
      "domain": "softwaregenesis.com",
      "name": "Software Genesis, Inc.",
      "organization_type": "Company",
      "organization_size": "Micro (0-9)",
      "country": "United States",
      "state": "Illinois",
      "district": NaN,
      "municipality": NaN,
      "summary": "Software Genesis is a software development company focused on the consumer market, offering a unique and revolutionary approach to software creation. They aim to partner with clients from the initial idea stage through to completion, guiding them through their processes. The company also indicates it may have its own product line available.",
      "summary_keywords": [
        "software development",
        "consumer market",
        "software company",
        "product development",
        "client p

## Exploring Query-Level Information

Each entry in the dataset represents a query and its associated search results.

So,
- Examining the fields available for each query
- Analyzing the number of results per query
- Extracting all queries for overview

Helps in understanding how the dataset is organized at a high level.

In [4]:
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)

# Understand top level structure
print(f"Type: {type(data)}")
print(f"Number of queries: {len(data)}")

# Look at first query
first = data[0]
print(f"\nFirst query: '{first['query']}'")
print(f"Total results: {first['total_results']}")
print(f"Number of results in data: {len(first['results'])}")

# Get all queries
queries = [(item['query_id'], item['query']) for item in data]
print(f"\nAll queries:")
for qid, q in queries:
    print(f"  {qid}: {q}")

Type: <class 'list'>
Number of queries: 101

First query: 'software companies'
Total results: 1000
Number of results in data: 1000

All queries:
  1: software companies
  2: healthcare providers
  3: AI startups in Berlin
  4: renewable energy companies in Germany
  5: companies providing cloud migration services
  6: B2B SaaS platforms
  7: machine learning companies
  8: IoT solution providers
  9: enterprise software vendors
  10: small consulting firms
  11: tech companies in Munich
  12: manufacturing in Baden-Württemberg
  13: e-commerce platforms
  14: pharmaceutical companies
  15: logistics companies in the US
  16: data analytics companies
  17: cybersecurity firms
  18: financial services companies
  19: retail chains in Europe
  20: construction companies
  21: marketing agencies
  22: human resources software
  23: supply chain management companies
  24: solar panel manufacturers
  25: agricultural technology companies
  26: electric vehicle manufacturers
  27: biotech com

## Transforming Nested Data into Tabular Format

The dataset contains nested structures where each query includes multiple results.

To enable analysis:
- Flatten the nested JSON structure
- Create two separate tables:
  - Query-level data (`queries_df`)
  - Result-level data (`results_df`)
- Add query metadata to each result for relational mapping

Clean the data to remove invalid characters and export it to Excel files.

In [5]:
# Load all results into a flat DataFrame
all_results = []
queries_df_list = []

for item in data:
    query_id = item['query_id']
    query = item['query']
    queries_df_list.append({'query_id': query_id, 'query': query})
    
    for result in item['results']:
        result['query_id'] = query_id
        result['query'] = query
        all_results.append(result)

# Two DataFrames — queries and results
queries_df = pd.DataFrame(queries_df_list)
results_df = pd.DataFrame(all_results)

print(f"Queries: {len(queries_df)}")
print(f"Total results: {len(results_df)}")
print(f"\nResults columns: {results_df.columns.tolist()}")
print(f"\nSample:")
print(results_df.head(3)[['query_id', 'query', 'rank', 'domain', 'similarity_score']])

ILLEGAL_CHARACTERS_RE = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')

def clean_excel_str(val):
    if isinstance(val, str):
        return ILLEGAL_CHARACTERS_RE.sub('', val)
    return val

# Save both
queries_df.to_excel('dataset/queries.xlsx', index=False)
results_df.applymap(clean_excel_str).to_excel('dataset/production_results.xlsx', index=False)
print("Saved!")

Queries: 101
Total results: 101000

Results columns: ['rank', 'similarity_score', 'domain', 'name', 'organization_type', 'organization_size', 'country', 'state', 'district', 'municipality', 'summary', 'summary_keywords', 'nace_code', 'query_id', 'query']

Sample:
   query_id               query  rank               domain  similarity_score
0         1  software companies     1  softwaregenesis.com          0.766201
1         1  software companies     2      standardpac.com          0.763732
2         1  software companies     3       nanosoft.co.za          0.760938


/scratch/ipykernel_26222/2583246381.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  results_df.applymap(clean_excel_str).to_excel('dataset/production_results.xlsx', index=False)


Saved!


## Summary

- The dataset consists of multiple queries, each with associated results
- Nested JSON structure was flattened into structured tables
- Clean datasets were exported for downstream analysis

These processed datasets can now be used for:
- Ranking evaluation
- Similarity analysis
- Model performance assessment